In [25]:
import pandas as pd
import numpy as np

In [27]:
df=pd.read_csv("titanic/train.csv")
df.head()
print(len(df))

891


In [28]:
# Missing value frequency per column
missing_freq = df.isnull().sum()
print("Missing value frequency per column:")
print(missing_freq)

# Drop columns where missing count > 443
cols_to_drop = missing_freq[missing_freq > 443].index.tolist()
print(f"\nColumns to drop (missing > 443): {cols_to_drop}")

df = df.drop(columns=cols_to_drop)
print(f"\nRemaining columns: {df.columns.tolist()}")

Missing value frequency per column:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Columns to drop (missing > 443): ['Cabin']

Remaining columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked']


In [29]:
#Step 1: Remove Duplicates
print(f"Duplicates before: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Duplicates after: {df.duplicated().sum()}")

Duplicates before: 0
Duplicates after: 0


In [30]:
# Step 2: Impute Missing Age with Median
df['Age'].fillna(df['Age'].median(), inplace=True)
print(f"Missing Age after impute: {df['Age'].isnull().sum()}")

Missing Age after impute: 0


C:\Users\SangaVarshaCloudangl\AppData\Local\Temp\ipykernel_22056\2648229965.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)


In [31]:
# Step 3: Impute Missing Embarked with Mode
print(f"Missing Embarked before: {df['Embarked'].isnull().sum()}")
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
print(f"Missing Embarked after: {df['Embarked'].isnull().sum}")

Missing Embarked before: 2
Missing Embarked after: <bound method Series.sum of 0      False
1      False
2      False
3      False
4      False
       ...  
886    False
887    False
888    False
889    False
890    False
Name: Embarked, Length: 891, dtype: bool>


In [32]:
freq_calculation = df.isnull().sum()
print(freq_calculation)

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


In [33]:
# Step 3: Fix Data Types
df['Pclass'] = df['Pclass'].astype('category')
df['Sex'] = df['Sex'].astype('category')
print(df.dtypes)


PassengerId       int64
Survived          int64
Pclass         category
Name             object
Sex            category
Age             float64
SibSp             int64
Parch             int64
Ticket           object
Fare            float64
Embarked         object
dtype: object


In [34]:
# Step 5: Cap Fare Outliers (IQR method)
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR
df['Fare'] = df['Fare'].clip(upper=upper)
print(f"Fare capped at: {upper:.2f}")

Fare capped at: 65.63


In [35]:
# Step 6: Drop unnecessary columns (Name, Ticket, PassengerId)
df = df.drop(columns=['Name', 'Ticket', 'PassengerId'], errors='ignore')
print(f"Remaining columns: {df.columns.tolist()}")

Remaining columns: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


In [36]:
# Step 7: Final Validation
print("--- Cleaned Dataset Info ---")
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())
print(f"\nShape: {df.shape}")
df.head()

--- Cleaned Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   Survived  891 non-null    int64   
 1   Pclass    891 non-null    category
 2   Sex       891 non-null    category
 3   Age       891 non-null    float64 
 4   SibSp     891 non-null    int64   
 5   Parch     891 non-null    int64   
 6   Fare      891 non-null    float64 
 7   Embarked  891 non-null    object  
dtypes: category(2), float64(2), int64(3), object(1)
memory usage: 43.9+ KB
None

Missing values:
Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

Shape: (891, 8)


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,65.6344,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [41]:
# Load and clean test data
test_df = pd.read_csv("titanic/test.csv")

# Merge Survived labels from gender_submission.csv
labels = pd.read_csv("titanic/gender_submission.csv")
test_df = test_df.merge(labels, on='PassengerId')

# Drop unnecessary columns
test_df = test_df.drop(columns=['Name', 'Ticket', 'PassengerId', 'Cabin'], errors='ignore')

# Impute missing
test_df['Age'] = test_df['Age'].fillna(test_df['Age'].median())
test_df['Embarked'] = test_df['Embarked'].fillna(test_df['Embarked'].mode()[0])
test_df['Fare'] = test_df['Fare'].fillna(test_df['Fare'].median())

print(f"Test shape: {test_df.shape}")
print(test_df.isnull().sum())

Test shape: (418, 8)
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
Survived    0
dtype: int64


In [42]:
# Encode categorical columns
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in ['Sex', 'Embarked']:
    df[col] = le.fit_transform(df[col].astype(str))
    test_df[col] = le.fit_transform(test_df[col].astype(str))

print("Encoding done")

Encoding done


In [43]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,1,22.0,1,0,7.2500,2
1,1,1,0,38.0,1,0,65.6344,0
2,1,3,0,26.0,0,0,7.9250,2
3,1,1,0,35.0,1,0,53.1000,2
4,0,3,1,35.0,0,0,8.0500,2


In [44]:
# Define features and target
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

# Use available columns only
features = [f for f in features if f in df.columns]

X_train = df[features]
y_train = df['Survived']

X_test = test_df[features]
y_test = test_df['Survived']

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (891, 7)
X_test shape: (418, 7)


In [45]:
# Train Random Forest Model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict on test data
y_pred = rf_model.predict(X_test)

# Evaluate
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.8206

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.85      0.86       266
           1       0.75      0.77      0.76       152

    accuracy                           0.82       418
   macro avg       0.81      0.81      0.81       418
weighted avg       0.82      0.82      0.82       418


Confusion Matrix:
[[226  40]
 [ 35 117]]
